# CS779 NMT — retrain English → Bengali / Hindi

Trains both language pairs from the preprocessed corpora and saves, for each one,
**a checkpoint and the vocabulary it was trained with**.

> The original competition run saved only the weights. Vocabularies were rebuilt in
> memory and lost, which left those checkpoints undecodable — embedding rows are
> addressed by index, so a vocabulary off by a single token turns output into noise.
> That is the failure this notebook exists to avoid.

**Before running:** turn on the GPU (Settings → Accelerator → **GPU T4 x2**) and attach
the datasets holding your preprocessed `.pkl` files.

Expect roughly 1–1.5 hours per language.

## 1. Clone the repo and install what's missing

In [ ]:
!git clone --depth 1 https://github.com/kunalchandra18/Neural-English-to-Indic-Machine-Translator.git /kaggle/working/nmt-repo

%cd /kaggle/working/nmt-repo
!pip install -q PyYAML

# Training reads pre-tokenized .pkl files, so spaCy and indic-nlp are NOT needed here.
# spaCy is installed only for the verification step, which tokenizes fresh English input.
!pip install -q spacy >/dev/null 2>&1
!python -m spacy download en_core_web_sm -q >/dev/null 2>&1

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE ⚠️")

## 2. Find the preprocessed data

Searches every attached dataset for the files produced by `preprocessing-code.ipynb`.
If anything is missing, fix the dataset attachment before continuing — do not proceed
with partial inputs.

In [ ]:
import glob, pickle

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return hits[0] if hits else None

PATHS = {
    "train":   find("train.pkl"),
    "bn_sent": find("preprocessed_testEnglish-Bengali_sentence.pkl"),
    "bn_ids":  find("preprocessed_testEnglish-Bengali_ids.pkl"),
    "hi_sent": find("preprocessed_testEnglish-Hindi_sentence.pkl"),
    "hi_ids":  find("preprocessed_testEnglish-Hindi_ids.pkl"),
}

missing = [k for k, v in PATHS.items() if v is None]
for k, v in PATHS.items():
    print(f"  {'OK ' if v else 'MISSING'}  {k:8s} {v or ''}")

if missing:
    raise SystemExit(
        f"\nMissing inputs: {missing}\n"
        "Attach the dataset(s) containing these files (Add Input, right-hand panel), "
        "or re-run scripts/preprocess.py on the raw competition JSON first."
    )

with open(PATHS["train"], "rb") as f:
    train = pickle.load(f)
need = {"source_Bengali", "target_Bengali", "source_Hindi", "target_Hindi"}
assert need <= set(train), f"train.pkl is missing keys: {need - set(train)}"
for lang in ("Bengali", "Hindi"):
    print(f"{lang}: {len(train[f'source_{lang}']):,} sentence pairs")

## 3. Write configs pointing at those paths

In [ ]:
from pathlib import Path
import yaml

BASE = dict(
    seq_length=55, min_freq=2, val_split=0.05, seed=42,
    emb_size=512, nhead=8, hid_dim=1024,
    num_encoder_layers=6, num_decoder_layers=6, dropout=0.15,
    epochs=15, train_batch_size=32, inference_batch_size=256,
    label_smoothing=0.1, weight_decay=0.01, warmup_steps=4000,
    max_grad_norm=1.0, bleu_sentences=40, compile_model=True,
)

for lang, code_, sent, ids in [("Bengali", "bn", "bn_sent", "bn_ids"),
                               ("Hindi",   "hi", "hi_sent", "hi_ids")]:
    cfg = dict(BASE, language=lang, lang_code=code_,
               train_pkl=PATHS["train"],
               test_sentences_pkl=PATHS[sent],
               test_ids_pkl=PATHS[ids],
               output_dir=f"/kaggle/working/runs/{lang.lower()}")
    Path(f"configs/kaggle_{code_}.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(f"wrote configs/kaggle_{code_}.yaml -> {cfg['output_dir']}")

## 4. Train Bengali

`--resume` picks up from the last checkpoint, so if the session dies you can re-run
this cell instead of starting over.

In [ ]:
!python scripts/train.py --config configs/kaggle_bn.yaml --resume

## 5. Train Hindi

In [ ]:
!python scripts/train.py --config configs/kaggle_hi.yaml --resume

## 6. Verify the saved artifacts — do not skip

Reloads each checkpoint **from disk** with its saved vocabulary and translates a fresh
sentence. This is the check that would have caught the original problem: if the pair
does not round-trip here, it will not work in the demo either.

In [ ]:
import pickle, sys
sys.path.insert(0, "/kaggle/working/nmt-repo/src")

import torch
from nmt.config import Config
from nmt.data import encode_corpus, make_loader
from nmt.decode import translate_loader
from nmt.model import build_model
from nmt.preprocessing import english

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLES = ["The weather is very pleasant today.",
           "She is reading a book in the library."]

ok = True
for code_ in ("bn", "hi"):
    cfg = Config.load(f"configs/kaggle_{code_}.yaml")
    vp, wp = f"{cfg.output_dir}/vocab_{code_}.pkl", cfg.best_model_path
    print(f"\n=== {cfg.language} ===")
    try:
        with open(vp, "rb") as f:
            v = pickle.load(f)
        src_vocab, tgt_vocab = v["src"], v["tgt"]
        model = build_model(cfg, len(src_vocab), len(tgt_vocab)).to(device)
        model.load_state_dict(torch.load(wp, map_location=device))
        model.eval()
        print(f"  vocab src={len(src_vocab):,} tgt={len(tgt_vocab):,}  (must match the checkpoint)")

        toks = english.tokenize_corpus(SAMPLES, n_process=1)
        loader = make_loader(encode_corpus(src_vocab, toks, cfg.seq_length), batch_size=2)
        for s, t in zip(SAMPLES, translate_loader(model, loader, tgt_vocab, cfg.seq_length, device)):
            print(f"  EN: {s}\n  ->  {t}\n")
    except Exception as e:
        ok = False
        print(f"  FAILED: {type(e).__name__}: {e}")

print("\n" + ("ALL LANGUAGES VERIFIED — safe to download" if ok else "VERIFICATION FAILED — do not deploy"))

## 7. Package the artifacts for download

Saves a half-precision copy as well. The fp32 checkpoint is ~270 MB per language;
fp16 halves that, which matters when uploading to a Space. Accuracy impact at
inference is negligible.

In [ ]:
import shutil, torch, os
from pathlib import Path

out = Path("/kaggle/working/nmt_release")
out.mkdir(exist_ok=True)

for code_, lang in (("bn", "bengali"), ("hi", "hindi")):
    src_dir = Path(f"/kaggle/working/runs/{lang}")
    dst = out / lang
    dst.mkdir(exist_ok=True)
    shutil.copy(src_dir / f"vocab_{code_}.pkl", dst)

    sd = torch.load(src_dir / f"best_model_{code_}.pth", map_location="cpu")
    torch.save(sd, dst / f"best_model_{code_}.pth")
    torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in sd.items()},
               dst / f"best_model_{code_}_fp16.pth")

for p in sorted(out.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(out)}  {p.stat().st_size/1e6:.0f} MB")

shutil.make_archive("/kaggle/working/nmt_release", "zip", out)
print(f"\nnmt_release.zip  {os.path.getsize('/kaggle/working/nmt_release.zip')/1e6:.0f} MB")
print("Download it from the Output panel on the right.")

## 8. Next: deploy the demo

1. Download `nmt_release.zip` and unzip it.
2. Create a Hugging Face Space — SDK **Gradio**, hardware **CPU basic** (free).
3. Upload from the repo: `app.py`, `src/`, `configs/bengali.yaml`, `configs/hindi.yaml`,
   and `requirements-demo.txt` **renamed to `requirements.txt`**.
4. Upload the weights so they land as:

   ```
   runs/bengali/best_model_bn.pth    runs/bengali/vocab_bn.pkl
   runs/hindi/best_model_hi.pth      runs/hindi/vocab_hi.pkl
   ```

   (If you use the fp16 files, rename them to drop the `_fp16` suffix.)

The Space builds and gives you the public link.